# M22 Discovery Entrypoint

Controlled notebook discovery entrypoint for Atlas DataFlow (M22-01).

This notebook accepts an explicit dataset input path, records discovery run parameters
and output locations, and enforces boundaries that prevent hidden training, release
publication, runtime mutation, or public production notebook execution.

## Usage

Run with an explicit `dataset_input_path`:

```
papermill notebooks/m22_discovery_entrypoint.ipynb output.ipynb \
    -p dataset_input_path /path/to/dataset.csv
```

Or set the parameters cell values before running the notebook locally.

## Boundaries

- This notebook is an **authoring and discovery surface only**.
- Notebook output is **not** the final operational source of truth.
- Downstream validation, contract derivation, model training, and publication are separate stages.
- Do not run this notebook in public production.
- Do not use implicit dataset paths or notebook state as inputs to later pipeline stages.

In [ ]:
# Parameters — supply all values explicitly; none may remain None at runtime.
dataset_input_path = None    # Required: path to the real dataset file (str)
run_id = None                # Optional: identifier for this discovery run (str or None)
output_dir = None            # Optional: directory for governed output artifacts (str or None)

In [ ]:
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

In [ ]:
# Guard: require explicit dataset_input_path; reject implicit or absent values.
if not dataset_input_path:
    raise ValueError(
        "dataset_input_path is required and must be an explicit path string. "
        "Do not rely on notebook state or implicit local paths."
    )

dataset_path = Path(dataset_input_path)
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {dataset_input_path}. "
        "Supply the path to a real, accessible dataset file."
    )

# Normalise optional parameters.
_run_id = run_id or f"discovery-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
_output_dir = Path(output_dir) if output_dir else dataset_path.parent / "discovery-output"
_output_dir.mkdir(parents=True, exist_ok=True)

print(f"dataset_input_path : {dataset_input_path}")
print(f"run_id             : {_run_id}")
print(f"output_dir         : {_output_dir}")

In [ ]:
# Side-effect boundary — these operations are explicitly forbidden in this entrypoint.
FORBIDDEN_SIDE_EFFECTS = {
    "model_training": False,
    "release_publication": False,
    "public_runtime_mutation": False,
    "public_production_notebook_execution": False,
    "public_dataset_upload": False,
    "automatic_business_objective_selection": False,
}
assert all(not v for v in FORBIDDEN_SIDE_EFFECTS.values()), (
    "Side-effect boundary violated; this entrypoint must not perform any forbidden operation."
)
print("Side-effect boundaries confirmed:", FORBIDDEN_SIDE_EFFECTS)

In [ ]:
# Record run parameters for traceability.
run_parameters = {
    "dataset_input_path": str(dataset_input_path),
    "run_id": _run_id,
    "output_dir": str(_output_dir),
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "dataset_sha256": hashlib.sha256(dataset_path.read_bytes()).hexdigest(),
}
print("Run parameters:", json.dumps(run_parameters, indent=2))

In [ ]:
# Discovery inspection — inspect the dataset and produce governed output artifacts.
# Extend this cell with real dataset inspection logic during authorized implementation.
discovery_summary = {
    "dataset_input_path": str(dataset_path),
    "file_size_bytes": dataset_path.stat().st_size,
    "discovery_note": "Stub — extend with real dataset inspection logic.",
}
print("Discovery summary:", json.dumps(discovery_summary, indent=2))

In [ ]:
# Record output locations — write governed discovery output artifact.
output_record_path = _output_dir / f"{_run_id}-discovery-record.json"
output_record = {
    "run_parameters": run_parameters,
    "discovery_summary": discovery_summary,
    "output_locations": {
        "discovery_record": str(output_record_path),
    },
    "notebook_boundary": "authoring_and_discovery_surface_only",
    "notebook_output_is_not_final_operational_truth": True,
}
output_record_path.write_text(json.dumps(output_record, indent=2), encoding="utf-8")
print(f"Output record written to: {output_record_path}")
print("Output locations:", json.dumps(output_record["output_locations"], indent=2))